# 01 — Clean & Merge
Loads outputs from `00_pull.ipynb`, cleans and merges PFAS, census, and Zillow data by ZIP code, and saves analysis-ready files to `data/cleaned_data/`.

**Inputs:** `data/01_pulls/`  
**Outputs:** `data/cleaned_data/final_map.csv`, `data/cleaned_data/final_clean_census.csv`

In [ ]:
# ── Imports ──────────────────────────────────────────────────────────────────
import os
import pandas as pd
import numpy as np
import geopandas as gpd
from pathlib import Path
from scipy import stats

# ── Helper functions ──────────────────────────────────────────────────────────
def find_repo_root(start=Path().resolve()):
    """Search upward for the repo root (directory containing README.md + data/)."""
    for parent in [start] + list(start.parents):
        if (parent / "README.md").exists() and (parent / "data").exists():
            return parent
    raise FileNotFoundError("Could not find repo root — make sure README.md and data/ exist.")

# ── Paths ─────────────────────────────────────────────────────────────────────
REPO_ROOT   = find_repo_root()
DATA_IN     = REPO_ROOT / "data" / "01_pulls"
DATA_OUT    = REPO_ROOT / "data" / "cleaned_data"
DATA_OUT.mkdir(parents=True, exist_ok=True)
os.chdir(REPO_ROOT)

print("Working directory:", os.getcwd())
print("Input:  ", DATA_IN)
print("Output: ", DATA_OUT)

## Loading Raw Data
Loading the raw files saved by `01_data_pull.ipynb` from the `data/` folder.

In [154]:
# Load UCMR5 occurrence data
ucmr_nj = pd.read_csv('data/01_pulls/ucmr5_nj_slim.csv')

ucmr_nj.head()

,PWSID,PWSName,State,Size,FacilityWaterType,Contaminant,AnalyticalResultValue,AnalyticalResultsSign,MRL,Units,CollectionDate,Region
0,NJ0102001,ATLANTIC CITY MUA,NJ,L,SW,PFUnA,NaN,<,0.002,µg/L,7/9/2024,2
1,NJ0102001,ATLANTIC CITY MUA,NJ,L,SW,ADONA,NaN,<,0.003,µg/L,7/9/2024,2
2,NJ0102001,ATLANTIC CITY MUA,NJ,L,SW,9Cl-PF3ONS,NaN,<,0.002,µg/L,7/9/2024,2
3,NJ0102001,ATLANTIC CITY MUA,NJ,L,SW,PFHpS,NaN,<,0.003,µg/L,7/9/2024,2
4,NJ0102001,ATLANTIC CITY MUA,NJ,L,SW,PFPeS,NaN,<,0.004,µg/L,7/9/2024,2


In [155]:
#Load ZIP codes data
nj_zc = pd.read_csv(
    'data/01_pulls/zipcodes_raw.csv',
    dtype={'PWSID': str, 'ZIPCode': str}
)

nj_zc.head()

,PWSID,ZIPCODE
0,010106001,6338
1,010109005,6382
2,020000005,13655
3,020000008,14070
4,020000008,14081


In [156]:
#Load Census race data
census_race_df = pd.read_csv(
    'data/01_pulls/census_race_raw.csv',
    dtype={'GEO_ID': str}
)

print(f'Census rows loaded: {len(census_race_df)}')
census_race_df.head()

Census rows loaded: 599


,GEO_ID,NAME,DP05_0001E,DP05_0001M,DP05_0002E,DP05_0002M,DP05_0003E,DP05_0003M,DP05_0004E,DP05_0004M,...,DP05_0104PM,DP05_0105PE,DP05_0105PM,DP05_0106PE,DP05_0106PM,DP05_0107PE,DP05_0107PM,DP05_0108PE,DP05_0108PM,Unnamed: 434
0,0400000US34,New Jersey,9343809,*****,4598393,534,4745416,534,96.9,0.1,...,0.1,(X),(X),6433714,(X),48.3,0.1,51.7,0.1,NaN
1,860Z200US07001,ZCTA5 07001,17121,1128,9860,747,7261,666,135.8,13.9,...,1.6,(X),(X),11991,(X),58.7,3.0,41.3,3.0,NaN
2,860Z200US07002,ZCTA5 07002,71553,58,35678,818,35875,816,99.5,4.5,...,0.8,(X),(X),47852,(X),48.5,1.4,51.5,1.4,NaN
3,860Z200US07003,ZCTA5 07003,53771,44,26660,617,27111,621,98.3,4.5,...,0.6,(X),(X),38709,(X),48.7,1.4,51.3,1.4,NaN
4,860Z200US07004,ZCTA5 07004,7824,317,3373,365,4451,339,75.8,12.6,...,0.9,(X),(X),5907,(X),44.3,5.0,55.7,5.0,NaN


In [157]:
#Loading census income data
census_income_df = pd.read_csv(
    'data/01_pulls/census_income_raw.csv',
    dtype={'GEO_ID': str}
)

print(f'Census rows loaded: {len(census_income_df)}')
census_income_df.head()

Census rows loaded: 599


,GEO_ID,NAME,S1901_C01_001E,S1901_C01_001M,S1901_C01_002E,S1901_C01_002M,S1901_C01_003E,S1901_C01_003M,S1901_C01_004E,S1901_C01_004M,...,S1901_C04_012M,S1901_C04_013E,S1901_C04_013M,S1901_C04_014E,S1901_C04_014M,S1901_C04_015E,S1901_C04_015M,S1901_C04_016E,S1901_C04_016M,Unnamed: 130
0,0400000US34,New Jersey,3507701,5116,4.1,0.1,2.7,0.1,4.8,0.1,...,554,83840,829,(X),(X),(X),(X),33.5,(X),NaN
1,860Z200US07001,ZCTA5 07001,5615,497,4.0,3.0,1.2,1.1,3.2,2.5,...,34319,82265,14811,(X),(X),(X),(X),50.1,(X),NaN
2,860Z200US07002,ZCTA5 07002,28951,793,4.9,1.1,4.7,1.2,6.0,1.3,...,4100,77800,11912,(X),(X),(X),(X),29.9,(X),NaN
3,860Z200US07003,ZCTA5 07003,21215,706,2.7,0.9,2.0,0.8,4.9,1.3,...,1825,108719,26584,(X),(X),(X),(X),26.1,(X),NaN
4,860Z200US07004,ZCTA5 07004,3039,272,6.3,5.7,0.0,1.5,2.5,2.9,...,64526,67047,17078,(X),(X),(X),(X),33.1,(X),NaN


In [ ]:
# Load Zillow Home Value Index (ZHVI) — December 2024
# ZHVI is a ZIP code-level measure and is used as-is here for ZIP-level analysis.
# For the water system proximity analysis (notebooks 03-04), an area-weighted
# version is computed via spatial overlay — see 03_clean_proximity.ipynb.
zhvi_df = pd.read_csv(
    'data/01_pulls/zhvi_nj.csv',
    dtype={'ZIPCODE': str}
)
zhvi_df['ZIPCODE'] = zhvi_df['ZIPCODE'].str.zfill(5)
zhvi_df['zhvi'] = pd.to_numeric(zhvi_df['zhvi'], errors='coerce')

print(f'Zillow rows loaded: {len(zhvi_df)}')
zhvi_df.head()

## Merging raw census race + income data

In [159]:
# Merge raw census race and income data on GEO_ID
# Both files have one row per ZIP code and share the same GEO_ID format (e.g. 860Z200US07001)
census_df = census_race_df.merge(
    census_income_df,
    on       = 'GEO_ID',
    how      = 'left',
    suffixes = ('', '_income')   # keep race columns as-is; income duplicates get _income suffix
)

print(f"Race rows:    {len(census_race_df)}")
print(f"Income rows:  {len(census_income_df)}")
print(f"Merged rows:  {len(census_df)}")
print(f"Columns:      {len(census_df.columns)}")
census_df.head()

Race rows:    599
Income rows:  599
Merged rows:  599
Columns:      565


,GEO_ID,NAME,DP05_0001E,DP05_0001M,DP05_0002E,DP05_0002M,DP05_0003E,DP05_0003M,DP05_0004E,DP05_0004M,...,S1901_C04_012M,S1901_C04_013E,S1901_C04_013M,S1901_C04_014E,S1901_C04_014M,S1901_C04_015E,S1901_C04_015M,S1901_C04_016E,S1901_C04_016M,Unnamed: 130
0,0400000US34,New Jersey,9343809,*****,4598393,534,4745416,534,96.9,0.1,...,554,83840,829,(X),(X),(X),(X),33.5,(X),NaN
1,860Z200US07001,ZCTA5 07001,17121,1128,9860,747,7261,666,135.8,13.9,...,34319,82265,14811,(X),(X),(X),(X),50.1,(X),NaN
2,860Z200US07002,ZCTA5 07002,71553,58,35678,818,35875,816,99.5,4.5,...,4100,77800,11912,(X),(X),(X),(X),29.9,(X),NaN
3,860Z200US07003,ZCTA5 07003,53771,44,26660,617,27111,621,98.3,4.5,...,1825,108719,26584,(X),(X),(X),(X),26.1,(X),NaN
4,860Z200US07004,ZCTA5 07004,7824,317,3373,365,4451,339,75.8,12.6,...,64526,67047,17078,(X),(X),(X),(X),33.1,(X),NaN


In [160]:
# Rename household income columns (C01) using the S1901 metadata
# Only keeping Households group — skipping Families, Married-couple, Nonfamily
# Only renaming Estimate (E) columns — skipping Margin of Error (M) columns
#Renaming pct race info
census_rename = {
    'S1901_C01_001E': 'hh_total',
    'S1901_C01_002E': 'hh_less_than_10k',
    'S1901_C01_003E': 'hh_10k_to_15k',
    'S1901_C01_004E': 'hh_15k_to_25k',
    'S1901_C01_005E': 'hh_25k_to_35k',
    'S1901_C01_006E': 'hh_35k_to_50k',
    'S1901_C01_007E': 'hh_50k_to_75k',
    'S1901_C01_008E': 'hh_75k_to_100k',
    'S1901_C01_009E': 'hh_100k_to_150k',
    'S1901_C01_010E': 'hh_150k_to_200k',
    'S1901_C01_011E': 'hh_200k_or_more',
    'S1901_C01_012E': 'hh_median_income',
    'S1901_C01_013E': 'hh_mean_income',

    'DP05_0033E'  : 'total_pop',
    'DP05_0083PE' : 'pct_white',
    'DP05_0084PE' : 'pct_black',
    'DP05_0085PE' : 'pct_native',
    'DP05_0086PE' : 'pct_asian',
    'DP05_0087PE' : 'pct_pacific_islander',
    'DP05_0090PE' : 'pct_hispanic',
    'DP05_0096PE' : 'pct_nonhispanic_white'
    
}

census_df = census_df.rename(columns=census_rename)

In [161]:
# Extract ZIPCODE from GEO_ID and keep only race + household income columns
census_df['ZIPCODE'] = census_df['GEO_ID'].str[-5:]

cols_to_keep = [
    # --- Identity ---
    'ZIPCODE',
    'NAME',
    # --- Race (from DP05) ---
    'total_pop',
    'pct_white',
    'pct_black',
    'pct_native',
    'pct_asian',
    'pct_pacific_islander',
    'pct_hispanic',
    'pct_nonhispanic_white',
    # --- Household income (from S1901 C01) ---
    'hh_total',
    'hh_less_than_10k',
    'hh_10k_to_15k',
    'hh_15k_to_25k',
    'hh_25k_to_35k',
    'hh_35k_to_50k',
    'hh_50k_to_75k',
    'hh_75k_to_100k',
    'hh_100k_to_150k',
    'hh_150k_to_200k',
    'hh_200k_or_more',
    'hh_median_income',
    'hh_mean_income',
]

cleaned_census_df = census_df[cols_to_keep].copy().drop(index=0).reset_index(drop=True)
cleaned_census_df

,ZIPCODE,NAME,total_pop,pct_white,pct_black,pct_native,pct_asian,pct_pacific_islander,pct_hispanic,pct_nonhispanic_white,...,hh_15k_to_25k,hh_25k_to_35k,hh_35k_to_50k,hh_50k_to_75k,hh_75k_to_100k,hh_100k_to_150k,hh_150k_to_200k,hh_200k_or_more,hh_median_income,hh_mean_income
0,07001,ZCTA5 07001,17121,45.1,34.4,1.4,17.0,0.0,25.3,27.9,...,3.2,4.5,11.0,9.7,15.0,22.6,9.7,19.0,101670,125172
1,07002,ZCTA5 07002,71553,65.5,17.3,1.5,10.2,0.1,32.1,42.5,...,6.0,6.7,8.2,14.5,14.8,19.5,8.2,12.6,83887,106298
2,07003,ZCTA5 07003,53771,56.5,20.1,1.7,11.0,0.0,30.9,37.4,...,4.9,5.5,5.6,14.8,12.6,20.0,12.6,19.1,103663,139365
3,07004,ZCTA5 07004,7824,94.7,1.8,0.2,4.2,0.2,11.7,82.2,...,2.5,4.5,3.6,5.0,16.0,23.8,15.1,23.1,115980,145876
4,07005,ZCTA5 07005,15307,86.6,5.3,1.5,8.7,0.2,11.6,74.1,...,3.7,5.2,3.7,7.1,11.8,17.6,16.1,28.6,132629,168824
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
593,08889,ZCTA5 08889,10193,90.4,1.4,0.3,8.2,0.0,8.9,79.2,...,1.9,3.3,3.8,16.0,7.4,10.8,16.0,36.1,157781,192863
594,08890,ZCTA5 08890,17,0.0,47.1,0.0,52.9,0.0,0.0,0.0,...,-,-,-,-,-,-,-,-,-,-
595,08901,ZCTA5 08901,57263,37.6,15.2,10.7,9.1,0.1,53.9,23.6,...,10.1,7.0,8.9,10.2,11.0,17.8,8.7,8.8,65810,87892
596,08902,ZCTA5 08902,43932,35.8,23.9,0.8,23.3,0.1,29.0,25.7,...,3.8,4.5,7.3,11.1,9.9,19.8,16.2,22.9,121071,141707


In [162]:
# pct_poc = all nonwhite races / total population = 100 - pct_white
cleaned_census_df['pct_white'] = pd.to_numeric(cleaned_census_df['pct_white'], errors='coerce')
cleaned_census_df['pct_poc'] = 100 - cleaned_census_df['pct_white']

print(cleaned_census_df[['ZIPCODE', 'pct_white', 'pct_poc']].head(10))

  ZIPCODE  pct_white  pct_poc
0   07001       45.1     54.9
1   07002       65.5     34.5
2   07003       56.5     43.5
3   07004       94.7      5.3
4   07005       86.6     13.4
5   07006       84.6     15.4
6   07008       34.7     65.3
7   07009       85.1     14.9
8   07010       63.5     36.5
9   07011       62.7     37.3


In [163]:
# Merge Zillow home values into cleaned census data
cleaned_census_df = cleaned_census_df.merge(
    zhvi_df[['ZIPCODE', 'zhvi']],
    on  = 'ZIPCODE',
    how = 'left'
)

print(f'ZIPs with ZHVI data:    {cleaned_census_df["zhvi"].notna().sum()}')
print(f'ZIPs missing ZHVI data: {cleaned_census_df["zhvi"].isna().sum()}')
print(cleaned_census_df[['ZIPCODE', 'zhvi']].head())

ZIPs with ZHVI data:    546
ZIPs missing ZHVI data: 52
  ZIPCODE           zhvi
0   07001  498164.455782
1   07002  578074.948462
2   07003  582757.329170
3   07004  770204.215553
4   07005  691455.433179


## Fixing Zipcodes and merging

In [164]:
#Merge UCMR5 occurrence data with zip codes
merged_ucmr_zc = ucmr_nj.merge(nj_zc, on='PWSID', how='left').copy()

print(f"Rows before join: {len(ucmr_nj)}")
print(f"Rows after join: {len(merged_ucmr_zc)}")
print(merged_ucmr_zc.head())

Rows before join: 56542
Rows after join: 263125
       PWSID            PWSName State Size FacilityWaterType Contaminant  \
0  NJ0102001  ATLANTIC CITY MUA    NJ    L                SW       PFUnA   
1  NJ0102001  ATLANTIC CITY MUA    NJ    L                SW       ADONA   
2  NJ0102001  ATLANTIC CITY MUA    NJ    L                SW  9Cl-PF3ONS   
3  NJ0102001  ATLANTIC CITY MUA    NJ    L                SW       PFHpS   
4  NJ0102001  ATLANTIC CITY MUA    NJ    L                SW       PFPeS   

   AnalyticalResultValue AnalyticalResultsSign    MRL Units CollectionDate  \
0                    NaN                     <  0.002  µg/L       7/9/2024   
1                    NaN                     <  0.003  µg/L       7/9/2024   
2                    NaN                     <  0.002  µg/L       7/9/2024   
3                    NaN                     <  0.003  µg/L       7/9/2024   
4                    NaN                     <  0.004  µg/L       7/9/2024   

   Region  ZIPCODE  
0    

## Selecting only PFAS values from UCMR5

In [166]:
print(merged_ucmr_zc["Contaminant"].unique())

['PFUnA' 'ADONA' '9Cl-PF3ONS' 'PFHpS' 'PFPeS' 'NFDHA' 'PFEESA' 'PFMBA'
 'PFPeA' 'PFMPA' '8:2 FTS' '4:2 FTS' '6:2 FTS' 'HFPO-DA' '11Cl-PF3OUdS'
 'PFTrDA' 'PFHxA' 'PFDoA' 'PFDA' 'PFOA' 'PFOS' 'PFNA' 'PFHxS' 'PFHpA'
 'PFBS' 'lithium' 'NMeFOSAA' 'NEtFOSAA' 'PFTA' 'PFBA']


In [167]:
#checking to see how many NaN values there are
print(merged_ucmr_zc["AnalyticalResultValue"].isna())

0         True
1         True
2         True
3         True
4         True
          ... 
263120    True
263121    True
263122    True
263123    True
263124    True
Name: AnalyticalResultValue, Length: 263125, dtype: bool


In [168]:
merged_ucmr_zc['AnalyticalResultValue'] = merged_ucmr_zc['AnalyticalResultValue'].fillna(0)

In [169]:
# Keep only PFAS (all start with "PF" or are HFPO-DA)
pfas_df = merged_ucmr_zc[
    merged_ucmr_zc['Contaminant'].str.startswith('PF') | 
    merged_ucmr_zc['Contaminant'].str.contains('HFPO-DA')
]

print(pfas_df['Contaminant'].unique())

['PFUnA' 'PFHpS' 'PFPeS' 'PFEESA' 'PFMBA' 'PFPeA' 'PFMPA' 'HFPO-DA'
 'PFTrDA' 'PFHxA' 'PFDoA' 'PFDA' 'PFOA' 'PFOS' 'PFNA' 'PFHxS' 'PFHpA'
 'PFBS' 'PFTA' 'PFBA']


In [170]:
#now I have to aggregate so its only one row for each zipcode... yay

zip_pfas = pfas_df.groupby('ZIPCODE').agg(
    max_pfas    = ('AnalyticalResultValue', 'max'),   # highest single PFAS reading
    mean_pfas   = ('AnalyticalResultValue', 'mean'),  # average across all PFAS
    total_detections = ('AnalyticalResultValue', lambda x: (x > 0).sum()),  # count of detections
    num_contaminants = ('Contaminant', 'nunique')     # how many different PFAS detected
).reset_index()

print(zip_pfas.head(10))

   ZIPCODE  max_pfas  mean_pfas  total_detections  num_contaminants
0     7002    0.0046   0.000164                 3                20
1     7003    0.0053   0.000080                 4                20
2     7004    0.0115   0.001414                17                20
3     7005    0.0111   0.000962                36                20
4     7006    0.0423   0.001141                69                20
5     7008    0.0073   0.000251                11                20
6     7009    0.0423   0.000506                15                20
7     7010    0.0140   0.001138                48                20
8     7011    0.0119   0.001979                23                20
9     7012    0.0119   0.001979                23                20


## Merging PFAS data with Census Data

In [171]:
# Standardize ZIPCODE format to 5-digit strings in both datasets
zip_pfas['ZIPCODE'] = zip_pfas['ZIPCODE'].astype(str).str.zfill(5)
cleaned_census_df['ZIPCODE'] = cleaned_census_df['ZIPCODE'].astype(str).str.zfill(5)

# Left join from census — keeps ALL census ZIP codes.
# ZIPs with no PFAS measurements will have NaN for PFAS columns.
final_census_df = cleaned_census_df.merge(
    zip_pfas,
    on  = 'ZIPCODE',
    how = 'left'
)

print(f"Census ZIPs:  {len(cleaned_census_df)}")
print(f"PFAS ZIPs:    {len(zip_pfas)}")
print(f"Merged rows:  {len(final_census_df)}")
print(f"ZIPs missing PFAS data: {final_census_df['max_pfas'].isna().sum()}")
final_census_df

Census ZIPs:  598
PFAS ZIPs:    516
Merged rows:  598
ZIPs missing PFAS data: 125


,ZIPCODE,NAME,total_pop,pct_white,pct_black,pct_native,pct_asian,pct_pacific_islander,pct_hispanic,pct_nonhispanic_white,...,hh_150k_to_200k,hh_200k_or_more,hh_median_income,hh_mean_income,pct_poc,zhvi,max_pfas,mean_pfas,total_detections,num_contaminants
0,07001,ZCTA5 07001,17121,45.1,34.4,1.4,17.0,0.0,25.3,27.9,...,9.7,19.0,101670,125172,54.9,498164.455782,NaN,NaN,NaN,NaN
1,07002,ZCTA5 07002,71553,65.5,17.3,1.5,10.2,0.1,32.1,42.5,...,8.2,12.6,83887,106298,34.5,578074.948462,0.0046,0.000164,3.0,20.0
2,07003,ZCTA5 07003,53771,56.5,20.1,1.7,11.0,0.0,30.9,37.4,...,12.6,19.1,103663,139365,43.5,582757.329170,0.0053,0.000080,4.0,20.0
3,07004,ZCTA5 07004,7824,94.7,1.8,0.2,4.2,0.2,11.7,82.2,...,15.1,23.1,115980,145876,5.3,770204.215553,0.0115,0.001414,17.0,20.0
4,07005,ZCTA5 07005,15307,86.6,5.3,1.5,8.7,0.2,11.6,74.1,...,16.1,28.6,132629,168824,13.4,691455.433179,0.0111,0.000962,36.0,20.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
593,08889,ZCTA5 08889,10193,90.4,1.4,0.3,8.2,0.0,8.9,79.2,...,16.0,36.1,157781,192863,9.6,691436.425854,0.0075,0.000254,19.0,20.0
594,08890,ZCTA5 08890,17,0.0,47.1,0.0,52.9,0.0,0.0,0.0,...,-,-,-,-,100.0,NaN,NaN,NaN,NaN,NaN
595,08901,ZCTA5 08901,57263,37.6,15.2,10.7,9.1,0.1,53.9,23.6,...,8.7,8.8,65810,87892,62.4,439987.676522,0.0077,0.000819,14.0,20.0
596,08902,ZCTA5 08902,43932,35.8,23.9,0.8,23.3,0.1,29.0,25.7,...,16.2,22.9,121071,141707,64.2,520066.172302,0.0000,0.000000,0.0,20.0


## Loading NJ Zipcode district shapes

In [ ]:
# Load NJ ZIP code boundaries from the GeoJSON saved by 00_pull.ipynb
nj_shapefile_slim = gpd.read_file('data/01_pulls/nj_zcta_2024.geojson')
print(f"NJ ZIP shapes: {len(nj_shapefile_slim)}")
print(nj_shapefile_slim.columns.tolist())

## Merging PFAS + Census data onto the shapefile

In [173]:

map_df = nj_shapefile_slim.merge(
    final_census_df,
    on   = 'ZIPCODE',
    how  = 'left'
)
#confirming all the data got merged properly
print(f"Rows in map data: {len(map_df)}")

Rows in map data: 598


In [174]:
# Convert all numeric columns to numbers
numeric_cols = [
    'max_pfas',
    'mean_pfas',
    'total_detections',
    'total_pop',
    'pct_white',
    'pct_black',
    'pct_asian',
    'pct_hispanic',
    'pct_native',
    'pct_pacific_islander',
    'pct_nonhispanic_white',
    'pct_poc',
    'hh_total',
    'hh_less_than_10k',
    'hh_10k_to_15k',
    'hh_15k_to_25k',
    'hh_25k_to_35k',
    'hh_35k_to_50k',
    'hh_50k_to_75k',
    'hh_75k_to_100k',
    'hh_100k_to_150k',
    'hh_150k_to_200k',
    'hh_200k_or_more',
    'hh_median_income',
    'hh_mean_income',
    'zhvi',
]

for col in numeric_cols:
    map_df[col] = pd.to_numeric(map_df[col], errors='coerce')

## Adding EPA standards for PFAS

In [175]:
#adding column for max values based on EPA standards
#in parts per billion already

# Flag ZIPs that exceed EPA MCL of 0.004 ug/L

map_df['exceeds_mcl_mean'] = map_df['mean_pfas'] > 0.004
map_df['exceeds_mcl_max'] = map_df['max_pfas'] > 0.004

print(map_df.columns.tolist())

['ZIPCODE', 'GEOID20', 'GEOIDFQ20', 'CLASSFP20', 'MTFCC20', 'FUNCSTAT20', 'ALAND20', 'AWATER20', 'INTPTLAT20', 'INTPTLON20', 'geometry', 'NAME', 'total_pop', 'pct_white', 'pct_black', 'pct_native', 'pct_asian', 'pct_pacific_islander', 'pct_hispanic', 'pct_nonhispanic_white', 'hh_total', 'hh_less_than_10k', 'hh_10k_to_15k', 'hh_15k_to_25k', 'hh_25k_to_35k', 'hh_35k_to_50k', 'hh_50k_to_75k', 'hh_75k_to_100k', 'hh_100k_to_150k', 'hh_150k_to_200k', 'hh_200k_or_more', 'hh_median_income', 'hh_mean_income', 'pct_poc', 'zhvi', 'max_pfas', 'mean_pfas', 'total_detections', 'num_contaminants', 'exceeds_mcl_mean', 'exceeds_mcl_max']


In [176]:
#saving back to the og data folder to visualize in another notebook
map_df.to_csv('data/cleaned_data/final_map.csv', index = False)
final_census_df.to_csv('data/cleaned_data/final_clean_census.csv', index=False)